# Semana 6 · Limpieza de datos climáticos — Estación Meteorológica del Fundo
## Inspección, limpieza y transformación básica de datos

**Objetivo técnico:** construir una versión limpia y trazable del histórico climático del fundo, sin inventar información ni borrar automáticamente observaciones que podrían ser válidas.

**Producto:** `clima_limpio.csv`

> Regla central: el archivo original se conserva intacto. Todo tratamiento se ejecuta sobre una copia y deja evidencia.

## Caso
La estación meteorológica del fundo (única para todo el predio) exportó el histórico climático del 01/03/2026 al 30/08/2026. El archivo declara extensión `.xls`, pero en realidad es una tabla HTML — un problema de formato común en exportaciones de plataformas agroclimáticas.

**Campos principales:** fecha, temperatura (media/mín/máx), humedad relativa, precipitación, ET0, viento, radiación solar, horas frío y grados-día del cultivo.

In [ ]:
# Si estás en Google Colab, esto abre el selector de archivos:
# sube el archivo Clima_sucio.xlsx cuando te lo pida (no importa el nombre exacto).
try:
    from google.colab import files
    uploaded = files.upload()
    ARCHIVO = next(iter(uploaded))
except ImportError:
    # Si NO estás en Colab (Jupyter local), coloca el archivo en la misma carpeta
    # que este notebook y ajusta el nombre aquí:
    ARCHIVO = "Clima_sucio.xlsx"

print("Archivo:", ARCHIVO)

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Archivo Excel real (ya convertido desde el HTML original exportado por la estación).
df_raw = pd.read_excel(ARCHIVO)

print("Dimensiones recibidas:", df_raw.shape)
display(df_raw.head(8))

## 1. Preservar el original
No trabajaremos directamente sobre `df_raw`. Creamos una copia de trabajo para que cualquier cambio sea reversible y auditable.

In [ ]:
df = df_raw.copy()

print("¿El original y la copia tienen las mismas dimensiones?", df_raw.shape == df.shape)

## 2. Inspección estructural antes de limpiar
Un perfilado inicial debe responder, como mínimo:

- ¿Cuántas filas y columnas existen?
- ¿Cómo se llaman los campos?
- ¿Qué tipos detecta Python?
- ¿Los valores numéricos tienen una magnitud razonable?
- ¿Hay fechas duplicadas?

In [ ]:
print("FILAS:", df.shape[0])
print("COLUMNAS:", df.shape[1])

print("\nENCABEZADOS:")
print(df.columns.tolist())

print("\nTIPOS DETECTADOS:")
print(df.dtypes)

print("\nFECHAS DUPLICADAS:")
print(df["Fecha"].duplicated().sum())

## 3. Normalizar los nombres de columnas
Usaremos nombres consistentes en minúsculas, sin tildes, espacios ni símbolos — esto no cambia los datos, solo la forma de referirnos a ellos.

In [ ]:
def normalizar_nombre_columna(nombre):
    nombre = str(nombre).strip().lower()
    nombre = "".join(
        ch for ch in unicodedata.normalize("NFKD", nombre)
        if not unicodedata.combining(ch)
    )
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    return nombre.strip("_")

df.columns = [normalizar_nombre_columna(c) for c in df.columns]

print(df.columns.tolist())

## 4. Convertir la fecha de forma controlada
El campo `fecha` llega como texto en formato `dd/mm/yyyy`. Lo convertimos explícitamente, sin dejar que pandas adivine el formato (evita ambigüedad día/mes).

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")

print(df["fecha"].dtype)
print("Fechas no interpretables:", df["fecha"].isna().sum())
print("Rango:", df["fecha"].min().date(), "->", df["fecha"].max().date())

df = df.sort_values("fecha").reset_index(drop=True)

## 5. Detectar y corregir el problema de escala (x100)
Al inspeccionar los valores numéricos, se detecta que **no tienen una magnitud biológicamente razonable**: una "temperatura" de 2272 no existe en la costa peruana. Comparando con los rangos esperados (temperatura ~15-35°C, humedad 0-100%), se concluye que el archivo exportó los decimales como enteros — es decir, todo viene **multiplicado por 100** (2272 → 22.72°C).

Esto no es un valor faltante ni un dato inventado: es un error de formato de exportación, verificado contra rangos físicos y corregido de forma explícita.

In [ ]:
columnas_escaladas = [
    "temperatura_oc", "temperatura_minima_oc", "temperatura_maxima_oc",
    "humedad_relativa", "humedad_relativa_minima", "humedad_relativa_maxima",
    "precipitacion_mm", "precipitacion_diaria_acumulada_mm",
    "et0_mm", "et0_acumulado_mm", "velocidad_de_viento_promedio_km_h",
    "radiacion_solar_w_m2", "grados_dia_oc", "grados_dia_acumulado_ano_oc",
]

# Nos quedamos solo con las que existan tal cual tras normalizar encabezados.
columnas_escaladas = [c for c in columnas_escaladas if c in df.columns]
print("Columnas a corregir:", columnas_escaladas)

for col in columnas_escaladas:
    df[col] = df[col] / 100

display(df[["fecha"] + columnas_escaladas[:4]].head(5))

## 6. Valores imposibles: marcar antes de corregir
Con la escala ya corregida, revisamos que los valores caigan en rangos físicamente posibles. Si algo quedara fuera de rango, se **marca con una bandera** en vez de eliminarse o reemplazarse por un número inventado.

In [ ]:
df["flag_temperatura_imposible"] = (
    df["temperatura_oc"].notna()
    & ((df["temperatura_oc"] < 0) | (df["temperatura_oc"] > 45))
)

df["flag_humedad_imposible"] = (
    df["humedad_relativa"].notna()
    & ((df["humedad_relativa"] < 0) | (df["humedad_relativa"] > 100))
)

columnas_flags = ["flag_temperatura_imposible", "flag_humedad_imposible"]
display(df[columnas_flags].sum().to_frame("registros_marcados"))

## 7. Duplicados exactos
Un duplicado exacto significaría que el mismo día fue exportado dos veces.

In [ ]:
duplicados_exactos = df.duplicated(subset=["fecha"], keep="first")
print("Duplicados exactos (misma fecha repetida):", duplicados_exactos.sum())

df = df.loc[~duplicados_exactos].copy()

## 8. Bitácora de limpieza
Documentamos qué problema se encontró, qué regla se aplicó y cuántos registros fueron afectados.

In [ ]:
bitacora = pd.DataFrame([
    {
        "etapa": "Estructura",
        "variable": "Encabezados",
        "problema": "Nombres con tildes, espacios y símbolos",
        "accion": "Normalizar a snake_case",
        "registros_afectados": len(df.columns),
        "justificacion": "Facilita un procesamiento reproducible.",
    },
    {
        "etapa": "Formato",
        "variable": "fecha",
        "problema": "Texto dd/mm/yyyy",
        "accion": "Convertir a datetime explícito",
        "registros_afectados": int(df["fecha"].notna().sum()),
        "justificacion": "Evita ambigüedad día/mes en fechas.",
    },
    {
        "etapa": "Escala",
        "variable": "Variables numéricas de clima",
        "problema": "Exportación multiplicó todos los valores por 100",
        "accion": "Dividir entre 100",
        "registros_afectados": len(df),
        "justificacion": "Magnitud verificada contra rangos físicos esperados para la zona.",
    },
    {
        "etapa": "Valores imposibles",
        "variable": "temperatura_oc, humedad_relativa",
        "problema": "Posibles valores fuera de rango físico",
        "accion": "Marcar con flag (no eliminar automáticamente)",
        "registros_afectados": int(df[columnas_flags].sum().sum()),
        "justificacion": "Preserva evidencia para revisión manual.",
    },
    {
        "etapa": "Duplicados",
        "variable": "fecha",
        "problema": "Filas con la misma fecha repetida",
        "accion": "Conservar la primera aparición",
        "registros_afectados": int(duplicados_exactos.sum()),
        "justificacion": "Evita doble conteo de un mismo día.",
    },
])

display(bitacora)

## 9. Resumen antes de exportar

In [ ]:
resumen = pd.DataFrame({
    "indicador": [
        "Filas recibidas",
        "Filas después de limpieza",
        "Duplicados exactos eliminados",
        "Temperaturas fuera de rango marcadas",
        "Humedades fuera de rango marcadas",
    ],
    "resultado": [
        len(df_raw),
        len(df),
        int(duplicados_exactos.sum()),
        int(df["flag_temperatura_imposible"].sum()),
        int(df["flag_humedad_imposible"].sum()),
    ],
})
display(resumen)

## 10. Exportar el producto
Se genera el dataset climático limpio, con tipos y escalas corregidos, listo para cruzarse con la evaluación fitosanitaria.

In [ ]:
df.to_csv("clima_limpio.csv", index=False, encoding="utf-8-sig")
bitacora.to_csv("bitacora_limpieza_clima.csv", index=False, encoding="utf-8-sig")

print("Archivos generados:")
print(" - clima_limpio.csv", df.shape)
print(" - bitacora_limpieza_clima.csv", bitacora.shape)

## Referencias técnicas
- pandas documentation — `read_html`, `to_datetime`, tipos de datos.
- Plan de Trabajo, Proyecto Productivo IB — Sistema de alerta temprana basado en grados-día para el control de la arañita roja en palto Hass, Fundo Loma Verde S.A.C.